## Reproducibility & AI-assistance disclosure

Part of `dislocation-speech-translation-fr-en`, companion to the M2 thesis *Dislocation under Translation: A Corpus Study of Whisper and XLS-R on Spontaneous Spoken French* (Zoé de Vries, Université Paris Cité, 2026; supervised by Prof. Ballier).

AI-assistance disclosure: explanatory comments and markdown were drafted with the help of a large language model (LLM) and reviewed by the author.

Pipeline order: `01_whisper` -> `02_xlsr` -> `03_align` -> `04_score`.

# Whisper — long-form transcription and translation (full recording)

Runs locally on a CUDA GPU.

1. Download the CFPP2000 source recording.
2. `transcribe` pass on the full audio -> pause-aligned segments. This segmentation is the canonical grid for the whole pipeline.
3. `translate` pass on the full audio -> continuous translation SRT.
4. For each canonical segment, slice the audio and run `translate` on the slice -> per-segment English translation, comparable with XLS-R.

Inter-notebook contract: this notebook writes `whisper_grid_manifest.csv` (`segment_id, t_start, t_end, dur_s, whisper_fr, whisper_en_per_seg`). Notebook 02 reads this file and applies the same segment boundaries.

Trade-off: the per-segment translation sees less context than the continuous pass; this is the price of segment-level comparability across models. The continuous pass is kept and re-indexed onto the grid as `whisper_en_cont` (last section).

In [ ]:
# 0. Environment.
import sys, site, os, pathlib, inspect, json, time, traceback
from pathlib import Path
import whisper, torch
from tqdm import tqdm

USER_SITE = site.getusersitepackages()
if os.path.isdir(USER_SITE) and USER_SITE not in sys.path:
    sys.path.insert(0, USER_SITE)

print("Python   :", sys.executable)
print("whisper  :", os.path.dirname(inspect.getfile(whisper)))
print("CUDA     :", torch.cuda.is_available())


In [ ]:
# 1. Config. The recording is downloaded at run time from the public CFPP2000 server
# and never redistributed.
SOURCE_URL = "http://cfpp2000.univ-paris3.fr/data/public/11eme/Anita_MUSSO_F_46_11e/Anita_MUSSO_F_46_11e.wav"
RAW_PATH   = Path("data/Anita_MUSSO_F_46_11e_raw.wav")
OUT_DIR    = Path("data/whisper_output")

# Whisper model; the transcribe pass defines the canonical segmentation.
MODEL_NAME = "medium"        # thesis variant
# MODEL_NAME = "large-v3"    # slower, more VRAM

LANG   = "fr"
SR     = 16_000
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUT_DIR / "whisper_grid_manifest.csv"
print("Model:", MODEL_NAME, "| Device:", DEVICE)
print("Manifest (input to notebook 02):", MANIFEST_PATH)


In [ ]:
# 2. Download the source WAV (idempotent) and load it 16 kHz mono.
import requests, librosa, numpy as np

if not RAW_PATH.exists():
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading source recording...")
    with requests.get(SOURCE_URL, stream=True, timeout=180) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(RAW_PATH, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 256):
                f.write(chunk); done += len(chunk)
                if total:
                    print(f"  {done/1024**2:.1f} / {total/1024**2:.1f} MB", end="\r")
    print("\nSaved:", RAW_PATH)
else:
    print("Source already present:", RAW_PATH)

print("Loading 16 kHz mono...")
audio_full, _ = librosa.load(RAW_PATH, sr=SR, mono=True)
print(f"  Total duration : {len(audio_full)/SR:.1f} s")


In [ ]:
# 3. Model + minimal SRT writer.
model = whisper.load_model(MODEL_NAME, device=DEVICE)

def save_srt(segments, out_path):
    def fmt(ts):
        h = int(ts // 3600); m = int((ts % 3600) // 60)
        s = int(ts % 60);    ms = int(round((ts - int(ts)) * 1000))
        return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"
    lines = []
    for i, seg in enumerate(segments, 1):
        lines += [str(i),
                  f"{fmt(seg.get('start', 0))} --> {fmt(seg.get('end', 0))}",
                  seg.get("text", "").strip(), ""]
    Path(out_path).write_text("\n".join(lines), encoding="utf-8")
    print("SRT written:", out_path)


In [ ]:
# 4. Transcribe pass -> canonical grid + SRT (word_timestamps=True for reliable boundaries;
# segments <= 0.05 s dropped).
print("Transcribe pass (full audio)...")
res_tr = model.transcribe(audio_full, language=LANG, task="transcribe",
                          word_timestamps=True, verbose=False)

segments = res_tr["segments"]
save_srt(segments, OUT_DIR / "Anita_MUSSO_transcribe.srt")

import csv
grid = []
for i, seg in enumerate(segments, 1):
    start, end = float(seg["start"]), float(seg["end"])
    if end - start <= 0.05:          # degenerate segment
        continue
    grid.append({
        "segment_id": f"seg_{i:04d}",
        "t_start": round(start, 3),
        "t_end":   round(end, 3),
        "dur_s":   round(end - start, 3),
        "whisper_fr": " ".join(seg.get("text", "").split()),
        "whisper_en_per_seg": "",
    })

print(f"{len(grid)} canonical segments.")
durs = [g["dur_s"] for g in grid]
print(f"Durations — min {min(durs):.2f}s | max {max(durs):.2f}s | mean {sum(durs)/len(durs):.2f}s")
print(f"Segments > 28 s (sub-chunked in notebook 02): {sum(d > 28 for d in durs)}")


In [ ]:
# 5. Continuous translate pass -> full-context SRT (re-indexed onto the grid in the last section).
print("Translate pass (full audio)...")
res_en = model.transcribe(audio_full, language=LANG, task="translate",
                          word_timestamps=True, verbose=False)
save_srt(res_en["segments"], OUT_DIR / "Anita_MUSSO_translate.srt")


In [ ]:
# 6. Per-segment translate: each slice translated in isolation -> whisper_en_per_seg;
# slices < 0.2 s left empty.
for g in tqdm(grid, desc="Whisper EN / segment"):
    a = int(g["t_start"] * SR); b = int(g["t_end"] * SR)
    clip = audio_full[a:b]
    if len(clip) < int(0.2 * SR):          # slice too short
        g["whisper_en_per_seg"] = ""
        continue
    try:
        out = model.transcribe(clip, language=LANG, task="translate", verbose=None)
        g["whisper_en_per_seg"] = " ".join((out.get("text", "") or "").split())
    except Exception as e:
        print("  ✗", g["segment_id"], e)
        g["whisper_en_per_seg"] = ""


In [ ]:
# 7. Write the shared manifest (read by notebook 02).
cols = ["segment_id", "t_start", "t_end", "dur_s", "whisper_fr", "whisper_en_per_seg"]
with open(MANIFEST_PATH, "w", encoding="utf-8", newline="") as f:
    wr = csv.DictWriter(f, fieldnames=cols)
    wr.writeheader()
    for g in grid:
        wr.writerow(g)

print("Manifest written:", MANIFEST_PATH, f"({len(grid)} rows)")
print("\n>>> MANUAL STEP: copy this file to the location notebook 02 reads as MANIFEST_PATH.")
for g in grid[:5]:
    print()
    print(g["segment_id"], f"[{g['t_start']}–{g['t_end']}]")
    print("  FR:", g["whisper_fr"][:120])
    print("  EN:", g["whisper_en_per_seg"][:120])


---
## Adding `whisper_en_cont` (no re-inference)

Run after the pipeline, or alone on a fresh kernel: reads the manifest and `Anita_MUSSO_translate.srt`, re-indexes the continuous translate pass onto the grid by temporal overlap, and rewrites the manifest. Idempotent.

In [ ]:
# Re-index the continuous SRT onto the grid -> whisper_en_cont. Idempotent.

import re as _re, pandas as pd
from pathlib import Path

SRT_PATH = OUT_DIR / "Anita_MUSSO_translate.srt"

def parse_srt(p):
    cues=[]
    for b in Path(p).read_text(encoding="utf-8").strip().split("\n\n"):
        ls=[l for l in b.splitlines() if l.strip()]
        if len(ls)<2: continue
        m=_re.match(r"(\d\d):(\d\d):(\d\d),(\d\d\d)\s*-->\s*(\d\d):(\d\d):(\d\d),(\d\d\d)", ls[1])
        if not m: continue
        g=list(map(int,m.groups()))
        s=g[0]*3600+g[1]*60+g[2]+g[3]/1000
        e=g[4]*3600+g[5]*60+g[6]+g[7]/1000
        cues.append((s,e," ".join(" ".join(ls[2:]).split())))
    return cues

cues=parse_srt(SRT_PATH)
df=pd.read_csv(MANIFEST_PATH)

def map_cont(s,e):
    hit=[(cs,t) for cs,ce,t in cues if t and min(e,ce)-max(s,cs)>0]
    hit.sort()
    return " ".join(t for _,t in hit)

df["whisper_en_cont"]=[map_cont(r.t_start,r.t_end) for r in df.itertuples()]
df.to_csv(MANIFEST_PATH,index=False,encoding="utf-8")

n=len(df)
print(f"{len(cues)} cues | {n} segments")
print("empty whisper_en_per_seg:", int(df.whisper_en_per_seg.fillna('').str.strip().eq('').sum()))
print("empty whisper_en_cont:", int(df.whisper_en_cont.fillna('').str.strip().eq('').sum()))
print("Manifest rewritten:", MANIFEST_PATH)
